# GHSL population — weeks 1–5

Population-only Bengaluru pilot. The Python scripts have been executed locally,
including real HDFS storage. This notebook is a walkthrough of the resulting files
and provides optional rerun commands; it is not an executed Colab session.

Open from the GHSL project with its `.venv` kernel. No Earth Engine authentication is
needed. Keep 2025/2030 projections separate from historical data. See README and
TEAM_HANDOFF for scope, sources and grid details.

In [ ]:
from pathlib import Path
import json, sys, subprocess
here = Path.cwd().resolve()
ROOT = next(p / 'population' for p in [here, *here.parents] if (p / 'population/config/pilot.json').exists())
print('Population project:', ROOT)
print(json.dumps(json.loads((ROOT / 'config/pilot.json').read_text()), indent=2))

## 1. Source provenance and grid alignment
Read the download manifest and the exact shared 1 km grid.

In [ ]:
manifest = json.loads((ROOT / 'reports/download_manifest.json').read_text())
print(len(manifest), 'validated source archives')
print(json.dumps(json.loads((ROOT / 'config/grid_contract.json').read_text()), indent=2))

## 2. Executed validation results
The reports below come from the actual local and HDFS Spark runs.

In [ ]:
for name in ['spark_validation.json', 'spark_hdfs_validation.json']:
    report = json.loads((ROOT / 'reports' / name).read_text())
    print(name, report['status'], report['input_rows'], 'input rows →', report['output_rows'], 'output rows')
print((ROOT / 'reports/tests.txt').read_text())

## 3. Query the delivered Parquet
Population is the sum of residents inside the pilot mask. Boundary cells may cover less than 1 km².

In [ ]:
import pyarrow.dataset as ds
dataset = ds.dataset(ROOT / 'data/curated/population_1km', format='parquet', partitioning='hive')
print(dataset.schema)
sample = dataset.to_table(filter=ds.field('year') == 2020).slice(0, 5)
sample.to_pylist()

In [ ]:
summary = json.loads((ROOT / 'reports/spark_hdfs_validation.json').read_text())['epochs']
for row in summary:
    print(row['year'], row['epoch_type'], f"{row['total_population']:,.3f}", 'estimated residents in pilot rectangle')

## 4. Optional pipeline rerun
Set RUN_PIPELINE to True only when you want to process the source files again. This uses a fresh output directory; existing outputs are not overwritten.

In [ ]:
RUN_PIPELINE = False
if RUN_PIPELINE:
    import uuid
    run_id = uuid.uuid4().hex[:8]
    for script in ['scripts/download_population_tiles.py', 'src/prepare_population_rasters.py']:
        subprocess.run([sys.executable, str(ROOT / script)], check=True)
    subprocess.run([sys.executable, str(ROOT / 'src/spark_population.py'),
                    '--output', str(ROOT / 'data/curated' / ('rerun_' + run_id)),
                    '--report', 'rerun_' + run_id + '.json'], check=True)
else:
    print('Using completed execution results; no recomputation requested.')

## 5. HDFS and team handoff
See README for HDFS start/stage/run/stop commands. The real HDFS execution is documented in reports/hdfs_fsck.txt and reports/spark_hdfs_validation.json. Join to teammates on (grid_id, cell_id, year) only after confirming grid and mask compatibility. The full three-product join has not been run because those inputs were not supplied.